In [4]:
import mne
import os
import numpy as np
import os.path as op
import matplotlib.pyplot as plt
import seaborn as sns
from mne.preprocessing import (ICA, create_ecg_epochs, annotate_muscle_zscore, annotate_movement, 
                               compute_average_dev_head_t, create_eog_epochs, maxwell_filter, find_bad_channels_maxwell, 
                               maxwell_filter_prepare_emptyroom, find_eog_events, find_ecg_events)
#source local
from mne import read_evokeds
from mne.minimum_norm import make_inverse_operator, apply_inverse_epochs, apply_inverse, write_inverse_operator, make_inverse_resolution_matrix, resolution_metrics, read_inverse_operator
# Download fsaverage from MNE-python
# mne.datasets.fetch_fsaverage(subjects_dir=None, verbose=None)

## Load paths and directories

In [5]:
# Set pathway and doucment location，you need transform file, brain anatomy 
s = 'p030' 
v = 'Final'
path = f'/Users/immlab/Desktop/IMM-Lab/Gabor_MEG'
meg_dir = op.join(path,'Prepro', f'{s}_MEG_ENS_Gabor')
Coregist_path = op.join(path,'Coregistration/MNE-fsaverage-data')
trans_file = op.join(meg_dir,v,'Source_Level',f'{s}-trans.fif')
#Info = mne.io.read_info(op.join(meg_dir,f'{s}_MEG_ENS_GABOR_raw.fif'))
# Parameters for stc
snr = 3.0
lambda2 = 1.0 / snr ** 2

## For checking STC file run this cell below

In [ ]:
# Read stc files if you save it before
# stc = mne.read_source_estimate(op.join(meg_dir,v,'Source_Level', f'{s}_90_deg_16gabor_stc-lh.stc')) # p014_0_deg_4gabor_stc-lh
#stc = mne.read_source_estimate(op.join(path,'Morph',s, f'{s}_0_deg_4gabor_Morphed-lh.stc')) # p014_0_deg_4gabor_stc-lh
stc = mne.read_source_estimate("C:/Users/csq2002.stu/Gabor_MEG/Prepro/p017_MEG_ENS_Gabor/Final/Source_Level/p017_0_deg_16gabor_control_stc-lh.stc")

initial_time = 0
brain_evoked = stc.plot(subject = s, 
                  hemi = 'split', 
                  subjects_dir=Coregist_path, 
                  initial_time=initial_time,
                  colormap = 'hot',
                #clim=dict(kind="value", lims=[3, 6, 9]),
                  smoothing_steps=5)
# Add title
brain_evoked.add_text(x=0.1, y=0.9, text='90_deg_16gabor', name="title", font_size=14)

In [ ]:
!jupyter nbextension enable --py widgetsnbextension --sys-prefix

## Co-registration

In [ ]:
# MNE co-registration, save the transform file by "-trans.fif"
mne.gui.coregistration(subjects_dir = Coregist_path)  
#pip install pyvistaqt & pip install ipywidgets pip install nibabel pip install darkdetect

Using pyvistaqt 3d backend.
    Triangle neighbors and vertex normals...
Using high resolution head model in /Users/immlab/Desktop/IMM-Lab/Gabor_MEG/Coregistration/MNE-fsaverage-data/P030/surf/lh.seghead
    Triangle neighbors and vertex normals...
Estimating fiducials from fsaverage.
    Triangle neighbors and vertex normals...
Using high resolution head model in /Users/immlab/Desktop/IMM-Lab/Gabor_MEG/Coregistration/MNE-fsaverage-data/P030/surf/lh.seghead
    Triangle neighbors and vertex normals...
Estimating fiducials from fsaverage.
Estimating fiducials from fsaverage.
Placing MRI fiducials - LPA
Using lh.seghead for head surface.
Placing MRI fiducials - LPA


Using high resolution head model in /Users/immlab/Desktop/IMM-Lab/Gabor_MEG/Coregistration/MNE-fsaverage-data/fsaverage/bem/fsaverage-head-dense.fif
    Triangle neighbors and vertex normals...
Using fiducials from: /Users/immlab/Desktop/IMM-Lab/Gabor_MEG/Coregistration/MNE-fsaverage-data/fsaverage/bem/fsaverage-fiducials.fif.
Loading MRI fiducials from /Users/immlab/Desktop/IMM-Lab/Gabor_MEG/Coregistration/MNE-fsaverage-data/fsaverage/bem/fsaverage-fiducials.fif... Done!
Using fsaverage-head-dense.fif for head surface.
    1 BEM surfaces found
    Reading a surface...
[done]
    1 BEM surfaces read
Using fsaverage-head-dense.fif for head surface.
    1 BEM surfaces found
    Reading a surface...
[done]
    1 BEM surfaces read
Channel types::	mag: 102, grad: 204
    Triangle neighbors and vertex normals...
Using high resolution head model in /Users/immlab/Desktop/IMM-Lab/Gabor_MEG/Coregistration/MNE-fsaverage-data/P030/surf/lh.seghead
    Triangle neighbors and vertex normals...
Estima

Traceback (most recent call last):
  File "/Users/immlab/Desktop/IMM-Lab/.venv/lib/python3.13/site-packages/mne/viz/backends/_qt.py", line 1744, in show
    self._widget.exec()
    ~~~~~~~~~~~~~~~~~^^
RuntimeError: wrapped C/C++ object of type QMessageBox has been deleted


/Users/immlab/Desktop/IMM-Lab/Gabor_MEG/Prepro/p030_MEG_ENS_Gabor/p030_2025_meg_ens_Gabor_raw-trans.fif transform file is saved.


In [4]:
# Load necessary files for below functions
empty_cov_esss = mne.read_cov(op.join(meg_dir,v,f'{s}_erm-cov.fif'))
evoked = mne.read_evokeds(op.join(meg_dir,v,'Evoked',f'{s}_epochs-ave.fif'))

# Load epochs data only for "apply_inverse_epochs"
#epochs = mne.read_epochs(op.join(meg_dir, v, f'{s}_epochs-epo.fif'))

    306 x 306 full covariance (kind = 1) found.
Reading /Users/immlab/Desktop/IMM-Lab/Gabor_MEG/Prepro/p030_MEG_ENS_Gabor/Final/Evoked/p030_epochs-ave.fif ...
    Found the data of interest:
        t =    -300.00 ...     800.00 ms (0_deg_1gabor)
        0 CTF compensation matrices available
        nave = 80 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.3, -0.05] s)
    Found the data of interest:
        t =    -300.00 ...     800.00 ms (0_deg_4gabor)
        0 CTF compensation matrices available
        nave = 80 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.3, -0.05] s)
    Found the data of interest:
        t =    -300.00 ...     800.00 ms (0_deg_4gabor_control)
        0 CTF compensation matrices available
        nave = 80 - aspect type = 100
No projector 

#### 'surfaces' can be scalp/skull/brain
1. scalp: one of ‘head’, ‘outer_skin’ (alias for ‘head’), ‘head-dense’, or ‘seghead’ (alias for ‘head-dense’)
2. skull: ‘outer_skull’, ‘inner_skull’, ‘brain’ (alias for ‘inner_skull’)
3. brain: one of ‘pial’, ‘white’, ‘inflated’, or ‘brain’ (alias for ‘pial’).

In [6]:
# Just to visualize the alignmnet of sensors, coils. Nothing really meaningful here.
# Load any file with information about the sensors and methods of measurement. 
Info = mne.io.read_info(op.join(meg_dir,f'{s}_MEG_ENS_GABOR_raw.fif'))

# Plot alignment to visually check, which plotting head, sensor, and source space alignment in 3D.
mne.viz.plot_alignment(info=Info, trans=trans_file, subject=s, dig=True,
    #mri_fiducials=True, so you should save fiducials for individual subject
    meg=["helmet", "sensors"], subjects_dir=Coregist_path, surfaces="head") 

    Read a total of 13 projection items:
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle


        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
Using pyvistaqt 3d backend.
Using outer_skin.surf for head surface.
Getting helmet for system 306m
Channel types::	mag: 102, grad: 204


## Generate BEM solution for a subject

In [7]:
# Make boundary element model (BEM) & solution
BEM_model = mne.make_bem_model(subject= s, ico=5, conductivity=(0.3,), subjects_dir= Coregist_path) #ico = Icosahedral spacing
bem_sol = mne.make_bem_solution(BEM_model)

# Save BEM model and solution file for later analysis
#mne.write_bem_surfaces(op.join(Coregist_path, s, 'bem', f'{s}-bem-model.fif'), BEM_model, overwrite=True)
mne.write_bem_solution(op.join(Coregist_path, s, 'bem', f'{s}-bem-solution.fif'), bem_sol, overwrite=True)

# Plot BEM, "Orientation": ‘coronal’ or ‘axial’ or ‘sagittal’.
#mne.viz.plot_bem(subject= s, subjects_dir=Coregist_path, brain_surfaces="white", orientation="coronal")  

Creating the BEM geometry...
Going from 5th to 5th subdivision of an icosahedron (n_tri: 20480 -> 20480)
inner skull CM is  -0.52 -20.06   6.74 mm
Surfaces passed the basic topology checks.
Complete.

Homogeneous model surface loaded.
Computing the linear collocation solution...
    Matrix coefficients...
        inner skull (10242) -> inner skull (10242) ...
    Inverting the coefficient matrix...
Solution ready.
BEM geometry computations complete.
Overwriting existing file.


In [3]:
# Plot BEM, "Orientation": ‘coronal’ or ‘axial’ or ‘sagittal’.
mne.viz.plot_bem(subject= s, subjects_dir=Coregist_path, brain_surfaces="white", orientation="coronal")  

NameError: name 'mne' is not defined

In [2]:
# Set up bilateral hemisphere surface-based source space, this is the grid within your boundary element model. 
src = mne.setup_source_space(subject=s, spacing="ico5", add_dist="patch", subjects_dir= Coregist_path) # spacing="ico5", oct = Octahedral spacing

# Save your source space file
src.save(op.join(meg_dir,v, 'Source_Level', f'{s}-ico-5-src.fif'), overwrite=True)
#src = mne.read_source_spaces(op.join(meg_dir,v, 'Source_Level', f'{s}-ico-5-src.fif'))

NameError: name 'mne' is not defined

In [ ]:
# Does not really matter
# Plot source space
# src.plot(subjects_dir=Coregist_path)

# # Plot BEM with source space, "Orientation": ‘coronal’ or ‘axial’ or ‘sagittal’.
# mne.viz.plot_bem(subject= s, subjects_dir=Coregist_path, orientation="axial",src=src) 

## Calculate a forward solution for a subject

In [1]:
# Forward solution: Brain sources → predicted sensor signals, 
# Describes how electrical currents in the brain generate magnetic fields detected by MEG sensors.
Forward_solution = mne.make_forward_solution(evoked[0].info, trans= trans_file, src=src, bem=bem_sol, meg=True)
mne.write_forward_solution(op.join(meg_dir,v,'Source_Level', f'{s}-fwd.fif'), Forward_solution, overwrite=True)
# Read inverse operator 
# Forward_solution = mne.read_forward_solution(op.join(meg_dir,v,'Source_Level', f'{s}-fwd.fif'))

NameError: name 'mne' is not defined

## Assemble inverse operator

In [11]:
# Compute inverse operator and save file 
Inverse = mne.minimum_norm.make_inverse_operator(info=evoked[0].info, 
                                                 forward=Forward_solution, 
                                                 noise_cov=empty_cov_esss, 
                                                 depth=0.7,
                                                 rank = "info") #Can specify depth weighting param.
inv_filename = op.join(meg_dir,v,'Source_Level', f'{s}-inv.fif')
write_inverse_operator(inv_filename, Inverse, verbose=True, overwrite=True)
# Read inverse operator 
# Inverse = mne.minimum_norm.read_inverse_operator(op.join(meg_dir,v,'Source_Level', f'{s}-inv.fif'))

Converting forward solution to surface orientation
    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]
Computing inverse operator with 306 channels.
    306 out of 306 channels remain after picking
Selected 306 channels
Creating the depth weighting matrix...
    204 planar channels
    limit = 19784/20484 = 10.004841
    scale = 1.89261e-08 exp = 0.7
Applying loose dipole orientations to surface source spaces: 0.2
Whitening the forward solution.
Computing rank from covariance with rank='info'
    MEG: rank 69 after 0 projectors applied to 306 channels
    Setting small MEG eigenvalues to zero (without PCA)
Creating the source covariance matrix
Adjusting source covariance matrix.
Computing SVD of whitened and weighted lead field matrix.
    largest singular value = 3.54788
    scaling factor to adjust the trace = 7.97528e+19 (nchan = 306 nzero = 237)
Write inverse operator decom

## For decoding

In [ ]:
epochs = mne.read_epochs(op.join(meg_dir, v, "Evoked", f'{s}_epochs-epo.fif'))
Inverse = mne.minimum_norm.read_inverse_operator(op.join(meg_dir,v,'Source_Level', f'{s}-inv.fif'))
stcs = apply_inverse_epochs(epochs,Inverse,
    lambda2= 1.0 / 3 ** 2,
    verbose=False,
    method="dSPM",
    pick_ori="normal",)

Reading /Users/immlab/Desktop/IMM-Lab/Gabor_MEG/Prepro/p030_MEG_ENS_Gabor/Final/Evoked/p030_epochs-epo.fif ...
    Found the data of interest:
        t =    -300.00 ...     800.00 ms
        0 CTF compensation matrices available
Not setting metadata
1974 matching events found
No baseline correction applied
0 projection items activated
Reading inverse operator decomposition from /Users/immlab/Desktop/IMM-Lab/Gabor_MEG/Prepro/p030_MEG_ENS_Gabor/Final/Source_Level/p030-inv.fif...
    Reading inverse operator info...
    [done]
    Reading inverse operator decomposition...
    [done]
    306 x 306 full covariance (kind = 1) found.
    Noise covariance matrix read.
    61452 x 61452 diagonal covariance (kind = 2) found.
    Source covariance matrix read.
    61452 x 61452 diagonal covariance (kind = 6) found.
    Orientation priors read.
    61452 x 61452 diagonal covariance (kind = 5) found.
    Depth priors read.
    Did not find the desired covariance matrix (kind = 3)
    Reading a sou

In [ ]:

# key press(motor, occipital) 5 and fixation 0 -> 1,2,3,4
# Visual 0.15s - sensation -> perception
# Apply the averaged epochs (Evoked data), via apply_inverse, choosing your own event.
stc = mne.minimum_norm.apply_inverse(evoked=evoked[0],
                                     inverse_operator=Inverse,
                                     lambda2=lambda2,
                                     verbose=False,
                                     method="dSPM")
                                    #  pick_ori="normal",)

In [ ]:
# Used previous particular averaged epochs events, to see its sources time course.
condition_name = evoked[0].comment
 # or '45', depending on your desired subject ID
initial_time = 0
brain_evoked = stc.plot(subject = s,
                  hemi = 'both',
                  subjects_dir=Coregist_path,
                  initial_time=initial_time,
                  colormap = 'hot',
                #clim=dict(kind="value", lims=[3, 6, 9]),
                  smoothing_steps=5)
# Add title
brain_evoked.add_text(x=0.1, y=0.9, text=condition_name, name="title", font_size=14)

In [ ]:
np.shape(epochs)

In [ ]:
raw_1000Hz = mne.io.Raw(op.join(meg_dir, v, f'{s}_GABOR_esss_ica_reject.fif'), preload=True)
# Resample the raw MEG data
raw = raw_1000Hz.resample(500,npad='auto')
# Find triggers
events = mne.find_events(raw, stim_channel='STI101', shortest_event=1/raw.info['sfreq'])

# Setting event name
event_id = {'0_deg_1gabor': 1,'0_deg_4gabor': 2,'0_deg_4gabor_control': 3,'0_deg_16gabor': 4,'0_deg_16gabor_control': 5,
    '45_deg_1gabor': 6,'45_deg_4gabor': 7,'45_deg_4gabor_control': 8,'45_deg_16gabor': 9,'45_deg_16gabor_control': 10,
    '90_deg_1gabor': 11,'90_deg_4gabor': 12,'90_deg_4gabor_control': 13,'90_deg_16gabor': 14,'90_deg_16gabor_control': 15,
    '135_deg_1gabor': 16,'135_deg_4gabor': 17,'135_deg_4gabor_control': 18,'135_deg_16gabor': 19,'135_deg_16gabor_control': 20}

In [ ]:
# Assume stcs is a list of SourceEstimate objects (n_epochs total)
n_epochs = len(stcs)
n_vertices, n_times = stcs[0].data.shape
# Create a 3D array: (n_epochs, n_vertices, n_times)
#X = np.stack([stc.data for stc in stcs], axis=0)  # shape: (n_epochs, n_vertices, n_times)
# Also get the times
times = stcs[0].times
y = epochs.events[:, 2]  # Or however you're assigning labels (from event_di

In [ ]:
src = mne.read_source_spaces(op.join(meg_dir,v, 'Source_Level', f'{s}-ico-5-src.fif'))

In [ ]:
import numpy as np
from collections import defaultdict
from itertools import combinations
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from mne import extract_label_time_course

# Parameters
window_size = 0.050  # 120 ms
step_size = 0.020    # 10 ms
sfreq = 500
win_samples = int(window_size * sfreq)
step_samples = int(step_size * sfreq)

n_reps = 10
n_conditions = 20
trials_per_condition = 80
trials_to_average = 4
n_avg_trials = trials_per_condition // trials_to_average

from mne import read_labels_from_annot

# Load all labels from the 'aparc' parcellation
all_labels = read_labels_from_annot(subject='p024',
                                    parc='aparc',
                                    subjects_dir=Coregist_path)

# Select only the labels you want by name
#label_names = ['fusiform', 'cuneus', 'pericalcarine', 'lingual', 'superiorparietal', 'inferiorparietal']
label_names = [ 'pericalcarine']  # ['cuneus']

labels = []
for region in label_names:
    lh = [lbl for lbl in all_labels if lbl.name == f"{region}-lh"]
    rh = [lbl for lbl in all_labels if lbl.name == f"{region}-rh"]
    if lh and rh:
        combined = lh[0] + rh[0]  # Adds labels geometrically
        combined.name = region
        labels.append(combined)


# Extract label time course: shape (n_trials, n_rois, n_times)
label_tc = extract_label_time_course(stcs, labels, src=src, mode='mean', return_generator=False)
label_tc = np.stack(label_tc)  # shape: (n_trials, n_rois, n_times)
n_trials, n_rois, n_times = label_tc.shape
start_idxs = np.arange(0, n_times - win_samples + 1, step_samples)
center_times = [times[start + win_samples // 2] for start in start_idxs]

# Crop time window from -0.050 to 0.600 seconds
tmin_crop, tmax_crop = -0.050, 0.600
time_mask = (times >= tmin_crop) & (times <= tmax_crop)

# Apply the mask
label_tc = label_tc[:, :, time_mask]
times = times[time_mask]

# Update decoding windows
n_times = label_tc.shape[-1]
start_idxs = np.arange(0, n_times - win_samples + 1, step_samples)
center_times = [times[start + win_samples // 2] for start in start_idxs]

# Group trial indices by condition
condition_indices = defaultdict(list)
for idx, label in enumerate(y):  # `y` = condition label for each trial
    condition_indices[label].append(idx)

# All unique pairs of conditions
label_pairs = list(combinations(sorted(condition_indices), 2))
n_pairs = len(label_pairs)

# For each ROI
for roi_idx, roi_name in enumerate(labels):
    print(f"ROI: {roi_name.name} ({roi_idx+1}/{len(labels)})")

    pairwise_scores = np.zeros((n_pairs, len(start_idxs), n_reps))

    for rep in range(n_reps):
        print(f"  Repetition {rep+1}/{n_reps}")

        # Average 4 trials per condition
        X_rep, y_rep = [], []
        for cond_label in sorted(condition_indices):
            trial_idxs = np.array(condition_indices[cond_label])
            np.random.shuffle(trial_idxs)
            for i in range(n_avg_trials):
                group = trial_idxs[i*trials_to_average:(i+1)*trials_to_average]
                avg_data = np.mean(label_tc[group, roi_idx, :], axis=0)  # shape: (n_times,)
                X_rep.append(avg_data)
                y_rep.append(cond_label)
        X_rep = np.stack(X_rep)  # shape: (n_conditions * n_avg_trials, n_times)
        y_rep = np.array(y_rep)

        # Pairwise decoding
        for p_idx, (label1, label2) in enumerate(label_pairs):
            mask = np.isin(y_rep, [label1, label2])
            X_pair = X_rep[mask]
            y_pair = y_rep[mask]
            y_pair = (y_pair == label2).astype(int)

            for i, start in enumerate(start_idxs):
                end = start + win_samples
                X_window = X_pair[:, start:end]  # shape: (n_epochs, win_samples)

                # Pipeline with PCA -> LDA
                clf = make_pipeline(
                    StandardScaler(),
                    PCA(n_components=0.99),
                    LinearDiscriminantAnalysis()
                )
                cv = StratifiedKFold(n_splits=5)
                scores = []
                for train_idx, test_idx in cv.split(X_window, y_pair):
                    clf.fit(X_window[train_idx], y_pair[train_idx])
                    y_pred = clf.predict(X_window[test_idx])
                    scores.append(accuracy_score(y_pair[test_idx], y_pred))
                pairwise_scores[p_idx, i, rep] = np.mean(scores)

    # Optionally save or return pairwise_scores here per ROI
    # e.g. np.save(f"decoding_{roi_name.name}.npy", pairwise_scores)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_single_pair(pairwise_scores, label_pairs, center_times, pair=None, pair_idx=None, roi_name=""):
    """
    Plot decoding accuracy over time for a specific condition pair.

    Parameters
    ----------
    pairwise_scores : np.ndarray
        Shape (n_pairs, n_time_windows, n_reps), decoding scores.
    label_pairs : list of tuple
        List of all condition label pairs (e.g., [(1,2), (1,3), ...]).
    center_times : array-like
        Time points corresponding to the center of each decoding window.
    pair : tuple, optional
        Condition label pair to plot (e.g., (1, 2)). Overrides `pair_idx`.
    pair_idx : int, optional
        Index of the pair in label_pairs.
    roi_name : str
        Region name (for figure title).
    """
    # Determine pair index
    if pair is not None:
        if pair not in label_pairs:
            raise ValueError(f"Pair {pair} not in label_pairs.")
        pair_idx = label_pairs.index(pair)
    elif pair_idx is None:
        raise ValueError("You must provide either `pair` or `pair_idx`.")

    # Extract and average scores
    scores = pairwise_scores[pair_idx]  # shape: (n_time_windows, n_reps)
    mean_scores = scores.mean(axis=1)
    std_scores = scores.std(axis=1)

    # Plot
    plt.figure(figsize=(8, 4))
    plt.plot(center_times, mean_scores, color='blue', label=f"{label_pairs[pair_idx][0]} vs {label_pairs[pair_idx][1]}")
    plt.fill_between(center_times, mean_scores - std_scores, mean_scores + std_scores, alpha=0.2, color='blue')
    plt.axhline(0.5, linestyle='--', color='gray', label='Chance')
    plt.title(f"ROI: {roi_name} – Pair {label_pairs[pair_idx][0]} vs {label_pairs[pair_idx][1]}")
    plt.xlabel("Time (s)")
    plt.ylabel("Decoding Accuracy")
    plt.ylim(0.4, 1.0)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Option 1: by label pair
plot_single_pair(pairwise_scores, label_pairs, center_times, pair=(1, 7), roi_name="pericalcarine")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_top_decoding_scores(pairwise_scores, label_pairs, roi_name):
    """
    Plot the top decoding accuracy for each condition pair in a given ROI.

    Parameters
    ----------
    pairwise_scores : np.ndarray
        Shape (n_pairs, n_time_windows, n_reps).
    label_pairs : list of tuples
        Each tuple contains the two condition labels (e.g., (1, 2)).
    roi_name : str
        Name of the brain ROI.
    """
    # Compute max accuracy across time and reps for each pair
    top_scores = pairwise_scores.max(axis=(1, 2))

    # Plot
    plt.figure(figsize=(12, 6))
    bars = plt.bar(range(len(top_scores)), top_scores, color='royalblue')
    plt.axhline(0.5, linestyle='--', color='gray', label='Chance Level')
    plt.xticks(range(len(top_scores)), [f'{a} vs {b}' for a, b in label_pairs], rotation=45, ha='right')
    plt.ylabel("Top Decoding Accuracy", fontsize=12)
    plt.title(f"Top Pairwise Decoding Accuracies in {roi_name}", fontsize=14)
    plt.ylim(0.4, 1.0)
    plt.tight_layout()
    plt.legend()
    plt.show()


In [ ]:
plot_top_decoding_scores(pairwise_scores, label_pairs, roi_name.name)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

def generate_all_pairs():
    """Generate list of all unique pairs for 20 conditions"""
    pairs = []
    for i in range(1, 21):  # Conditions 1-20
        for j in range(i + 1, 21):  # Only pairs where j > i
            pairs.append((i, j))
    return pairs

def plot_pairwise_in_batches(pairwise_scores, label_pairs, center_times, 
                           roi_name="pericalcarine", batch_size=20, 
                           save_path="C:/Users/csq2002.stu/Gabor_MEG/Prepro/p026_MEG_ENS_Gabor/Pairs"):
    """
    Plot all pairwise comparisons in batches and save to specified path
    """
    
    # Create save directory if it doesn't exist
    os.makedirs(save_path, exist_ok=True)
    print(f"Saving figures to: {save_path}")
    
    all_pairs = generate_all_pairs()
    n_pairs = len(all_pairs)
    
    print(f"Plotting {n_pairs} pairs in batches of {batch_size}")
    
    # Calculate number of batches
    n_batches = (n_pairs + batch_size - 1) // batch_size
    
    for batch_idx in range(n_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, n_pairs)
        batch_pairs = all_pairs[start_idx:end_idx]
        
        print(f"\nBatch {batch_idx + 1}/{n_batches}: Plotting pairs {start_idx + 1} to {end_idx}")
        
        # Create subplot grid
        n_pairs_batch = len(batch_pairs)
        cols = 4  # 4 plots per row
        rows = (n_pairs_batch + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(16, 4*rows))
        
        # Handle single plot case
        if n_pairs_batch == 1:
            axes = np.array([axes])
        elif rows == 1:
            axes = axes.reshape(1, -1)
        
        for idx, pair in enumerate(batch_pairs):
            row = idx // cols
            col = idx % cols
            
            if rows == 1:
                ax = axes[0, col] if cols > 1 else axes[col]
            else:
                ax = axes[row, col]
            
            print(f"  Plotting: {pair[0]} vs {pair[1]}")
            
            # Set the current subplot as active
            plt.sca(ax)
            
            # ACTUAL PLOTTING HAPPENS HERE (without ax parameter)
            plot_single_pair(pairwise_scores, label_pairs, center_times, 
                           pair=pair, roi_name=roi_name)
            
            ax.set_title(f"Pair {pair[0]} vs {pair[1]}", fontsize=8)
        
        # Turn off empty subplots
        total_subplots = rows * cols
        for idx in range(n_pairs_batch, total_subplots):
            row = idx // cols
            col = idx % cols
            if rows == 1:
                ax = axes[0, col] if cols > 1 else axes[col]
            else:
                ax = axes[row, col]
            ax.axis('off')
        
        plt.suptitle(f'Batch {batch_idx + 1}: Pairwise Comparisons - {roi_name}', fontsize=12)
        plt.tight_layout()
        
        # Save batch to your specified path
        save_filename = os.path.join(save_path, f'pairwise_batch_{batch_idx + 1:02d}_{roi_name}.png')
        plt.savefig(save_filename, dpi=150, bbox_inches='tight')
        print(f"  Saved: {save_filename}")
        plt.show()

# TO USE THIS FUNCTION:
# plot_pairwise_in_batches(pairwise_scores, label_pairs, center_times, roi_name="pericalcarine", batch_size=20)

# The figures will be automatically saved to: C:/Users/csq2002.stu/Gabor_MEG/Prepro/p026_MEG_ENS_Gabor/Pairs

In [ ]:
plot_pairwise_in_batches(pairwise_scores, label_pairs, center_times, roi_name="pericalcarine", batch_size=20)

# Visualization on source time courses data in whole brian

In [ ]:
# Generate source time courses file on evoked data
condition = 17
condition_name = evoked[condition].comment
stc = mne.minimum_norm.apply_inverse(evoked=evoked[condition],
                                     inverse_operator=Inverse,
                                     lambda2=lambda2,
                                     verbose=False,
                                     method="dSPM")       
# Plot the source estimates
initial_time = 0.1
brain_evoked = stc.plot(subject=s,
            hemi='both',
            subjects_dir=Coregist_path,
            initial_time=initial_time,
            colormap='hot',
            smoothing_steps=5)
        
# Add title with condition name
brain_evoked.add_text(x=0.1, y=0.9, text=condition_name, name="title", font_size=14)

# Save stc for the specific event if necessary
#stc.save(op.join(meg_dir, v, f'{s}_{condition_name}_stc'), overwrite=True)

In [ ]:
# Loop through all 22 conditions and save these files
for condition in range(22):
    try:
        # Apply inverse operator to the evoked data
        stc = mne.minimum_norm.apply_inverse(
            evoked=evoked[condition],
            inverse_operator=Inverse,
            lambda2=lambda2,
            verbose=False,
            method="dSPM")
        
        # Get condition name from evoked data
        condition_name = evoked[condition].comment
        # Save the STC file
        stc.save(op.join(meg_dir, v, 'Source_Level', f'{s}_{condition_name}_stc'), overwrite=True)
        
        print(f"Condition {condition} ({condition_name}) processed and saved")
        
    except Exception as e:
        print(f"Error processing condition {condition}: {e}")

print("All conditions processed")

In [ ]:
# Extract time courses from space sources
region = "lateraloccipital-lh"

# Read the label for the specified region
label = mne.read_labels_from_annot(subject=s, parc='aparc', regexp=region, subjects_dir=Coregist_path)

# Extract time courses for the region
stc_label = mne.extract_label_time_course(stc, label[0], src=Forward_solution['src'], mode=None, return_generator=False)[0] #inverse['src']

# plot the times series of  label
fig, axes = plt.subplots(1, layout="constrained")
axes.plot(stc.times, stc_label[0], "k")
axes.set(xlabel="Time (ms)", ylabel="MNE AU")
axes.legend()

## Morph stcs across all participants (Max's code)

In [ ]:
for s in subs:
    print(s)
    #here is where I need the correct source space for each subject
    fwd = mne.read_forward_solution(op.join(meg_dir,v,'Source_level',f'{s}_fwd-fif'), verbose=False)
    src = fwd['src']
    morph = mne.compute_source_morph(
            src = src,
            subject_from=f"vml_mri_{s}",
            subject_to = 'vml_avg_child',
            src_to=src_to, #fsaverage's volumetric src
            subjects_dir=mri_path,
            # niter_sdr=[5, 5,  2],
            # niter_affine=[5, 5, 2],
            verbose=False
        )
    morph_mat = morph.compute_vol_morph_mat()
    morph.save(op.join(meg_path, f'vml_meg_{s}', f'{s}-morph'), overwrite=True)
    morph = mne.read_source_morph(op.join(meg_path, f'vml_meg_{s}', f'{s}-morph-morph.h5'))
    for cond in conditions:
        stc = mne.read_source_estimate(op.join(meg_path, f'vml_meg_{s}', f'DS_{s}_cond-{cond}-stc.h5'))
        morphed_stc = morph.apply(stc)
        morphed_stc.save(op.join(meg_path, f'vml_meg_{s}', f'morphed_DS_{s}_cond-{cond}-stc.h5'), overwrite=True)

## Source time courses for Machine Learning & time-freqency analysis using individual epoch

In [ ]:
# Set pathway and doucment location，you need transform file, brain anatomy 
s = 'p016' 
v = 'Final'
path = f'D:/MEG_CODE/MEG_Gabor'
meg_dir = op.join(path,'Prepro', f'{s}_MEG_ENS_Gabor')
Coregist_path = op.join(path,'Coregisteration')
trans_file = op.join(path,v,f'{s}_MEG_ENS_Gabor',f'{s}_trans.fif')
#Info = mne.io.read_info(op.join(meg_dir,f'{s}_MEG_ENS_GABOR_raw.fif'))
# Parameters for stc
snr = 3.0
lambda2 = 1.0 / snr ** 2

In [ ]:
# Read epoched data
epochs = mne.read_epochs(op.join(meg_dir,v,"Evoked","p016_epochs-epo.fif"), preload=True)
Inverse = mne.minimum_norm.read_inverse_operator('D:\MEG_CODE\MEG_Gabor\Prepro\p016_MEG_ENS_Gabor\Final\Source_Level\p016-inv.fif')

In [ ]:
# Method 2
# Apply the inverse operator to the single trial epochs for decodingh MEG data in source space
# Give you source estimates for each individual condition trial, which you can then analyze for trial-to-trial variability
STCs = mne.minimum_norm.apply_inverse_epochs(epochs,inverse_operator=Inverse,lambda2=lambda2,
                                             verbose=False, method="dSPM")
# initial_time = 0.1
# brain_STCs = STCs.plot(subject = s, 
#                   hemi = 'both', 
#                   subjects_dir=Coregist_path, 
#                   initial_time=initial_time,
#                 #clim=dict(kind="value", lims=[3, 6, 9]),
#                   smoothing_steps=10)
#STCs.save(op.join(meg_dir, v,'Source_Level', f'{s}_epoch_Trial.stc'), overwrite=True)